# Transfer Learning for Biomedical Imaging
Tunneling Nanotubes (TNT) Detection in Cancer Cells

# Introduction

## Information About the Project

### Objective
The primary objective of this project is to apply transfer learning to detect tunneling nanotubes (TNTs) in fluorescence microscopy images of cancer cells. This is a binary patch classification task, where the model predicts whether a given image patch contains a TNT or not.

### Scope
The scope of the project focuses on building a complete, working end-to-end pipeline rather than achieving the highest possible accuracy. Key steps include:

* Extracting labeled image patches from a pair of provided large images.

* Building a custom dataset for binary classification.  

* Applying a transfer learning model (such as VGG16) to the data.  

* Training models on two different patch sizes (512 px and 256 px) and comparing their performance.

* Implementing checkpointing and evaluating a three-phase training schedule.

## Description of the Dataset

* Source: The data consists of fluorescence microscopy images. These images are captured from cancer cells that have been grown on a glass slide, labeled with fluorescent markers, and scanned using a fluorescence microscope. The dataset structure contains exactly two identical large images: original.png (the clean image) and annotated.png (the image with TNTs manually marked by yellow lines).

* Size: The original files are very large images comprising thousands of pixels per dimension. From these, the dataset will be generated by extracting smaller overlapping or non-overlapping patches, specifically sized at 512x512 pixels and 256x256 pixels.

* Type: This is image data (microscopy images).  

## Description of the Columns

* Target Variable: The target is a discrete binary label indicating the presence of a tunneling nanotube. The label is assigned as 1 if the patch contains a TNT (determined by the presence of yellow lines in the corresponding patch of the annotated image) and 0 if it contains no TNT. The dataset features a class imbalance, as most patches contain no TNTs.

* Feature Variables: The features are the image patches extracted exclusively from the original.png clean image. These features take the form of image pixel arrays (either 512x512 or 256x256 pixels)

# Task 1: Setup and Patch Creation

## Step 1.1: Import Libraries and Set Seeds

In [8]:
import os
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import copy
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
seed = 42
set_seed(seed)

# Configure device for GPU/CPU use
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Step 1.2: (Optional) Shadow Removal

## Step 1.3: Patch Extraction

In [5]:
def extract_patches(original_path, annotated_path, patch_size, stride, output_dir):
    """
    Extracts patches from the original image and sorts them into 'tnt' or 'no_tnt' folders
    based on the presence of yellow TNT annotations in the annotated image.
    """
    # Output directories for the two classes
    tnt_dir = os.path.join(output_dir, 'tnt')
    no_tnt_dir = os.path.join(output_dir, 'no_tnt')
    os.makedirs(tnt_dir, exist_ok=True)
    os.makedirs(no_tnt_dir, exist_ok=True)

    # Load both the original and annotated images
    original_img = cv2.imread(original_path)
    annotated_img = cv2.imread(annotated_path)

    if original_img is None or annotated_img is None:
        print("Error: Could not load one or both images. Please check the file paths.")
        return

    # Get image dimensions (assuming both images are the same size)
    height, width, _ = original_img.shape

    # Define the HSV (hue, saturation, and value) range for the yellow color
    lower_yellow = np.array([20, 100, 100])
    upper_yellow = np.array([40, 255, 255])

    patch_count = 0
    tnt_count = 0
    no_tnt_count = 0

    # Slide a window across the image using the specified patch_size and stride
    for y in range(0, height - patch_size + 1, stride):
        for x in range(0, width - patch_size + 1, stride):
            
            # Crop the patches from both images
            orig_patch = original_img[y:y + patch_size, x:x + patch_size]
            ann_patch = annotated_img[y:y + patch_size, x:x + patch_size]

            # Convert the annotated patch to HSV
            hsv_patch = cv2.cvtColor(ann_patch, cv2.COLOR_BGR2HSV)

            # Create a mask to detect the yellow lines
            mask = cv2.inRange(hsv_patch, lower_yellow, upper_yellow)

            # Check if there are any yellow pixels (TNTs) in the mask
            if cv2.countNonZero(mask) > 0:
                # Label = 1 (Contains TNT)
                save_path = os.path.join(tnt_dir, f"patch_{patch_count}.png")
                tnt_count += 1
            else:
                # Label = 0 (No TNT)
                save_path = os.path.join(no_tnt_dir, f"patch_{patch_count}.png")
                no_tnt_count += 1

            # Save the patch strictly from the ORIGINAL image
            cv2.imwrite(save_path, orig_patch)
            patch_count += 1

    print(f"Extraction complete for {output_dir}:")
    print(f"  Total patches: {patch_count}")
    print(f"  TNT patches (label 1): {tnt_count}")
    print(f"  No TNT patches (label 0): {no_tnt_count}")


if __name__ == "__main__":
    # Define file paths
    orig_path = "../../data/tnt/m05.png"
    ann_path = "../../data/tnt/m05-label.png"

    # Run 1: 512x512 patches with stride 256
    print("Processing 512x512 patches...")
    extract_patches(
        original_path=orig_path,
        annotated_path=ann_path,
        patch_size=512,
        stride=256,
        output_dir="processed_512"
    )

    # Run 2: 256x256 patches with stride 128
    print("\nProcessing 256x256 patches...")
    extract_patches(
        original_path=orig_path,
        annotated_path=ann_path,
        patch_size=256,
        stride=128,
        output_dir="processed_256"
    )

Processing 512x512 patches...
Extraction complete for processed_512:
  Total patches: 391
  TNT patches (label 1): 142
  No TNT patches (label 0): 249

Processing 256x256 patches...
Extraction complete for processed_256:
  Total patches: 1680
  TNT patches (label 1): 217
  No TNT patches (label 0): 1463


# Task 2: Custom Dataset and DataLoaders

## Step 2.1: TNTDataset Class

In [6]:
class TNTDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Custom dataset for TNT binary classification.
        
        Args:
            root_dir (str): Path to the processed directory
            transform (callable, optional): Optional transform to be applied on a sample
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Define the label mapping based on the processed subfolder names
        class_mapping = {'no_tnt': 0, 'tnt': 1}

        # Traverse the directories to collect image paths and their corresponding labels
        for class_name, label in class_mapping.items():
            class_dir = os.path.join(root_dir, class_name)
            
            # Skip if the directory doesn't exist yet
            if not os.path.isdir(class_dir):
                print(f"Warning: Directory not found -> {class_dir}")
                continue
                
            for file_name in os.listdir(class_dir):
                # Ensure we only read image files
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, file_name))
                    self.labels.append(label)

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.image_paths)

    def __getitem__(self, idx):
        """Generates one sample of data."""
        img_path = self.image_paths[idx]
        
        # Load the image using PIL
        image = Image.open(img_path).convert('RGB')
        
        # Get the corresponding label and convert it to a torch float tensor
        # (BCEWithLogitsLoss expects float labels for binary classification)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Apply any ImageNet-normalized transforms or augmentations
        if self.transform:
            image = self.transform(image)

        return image, label

## Step 2.2: Transforms and Splits

In [9]:
# ImageNet normalized transforms
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Training transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


# Validation/Test transforms (NO augmentation)
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


# Helper class to apply transforms to PyTorch Subsets
class TransformSubset(Dataset):
    """
    Wrapper to apply a specific transform to a PyTorch Subset.
    This allows us to use different transforms for train and val/test splits.
    """
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
            return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset.dataset[self.subset.indices[index]]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

def create_dataloaders(root_dir):
    """
    Creates stratified 70/15/15 splits and returns DataLoaders for a given processed directory.
    """
    # Instantiate the base dataset WITHOUT transforms so we can extract PIL images first
    base_dataset = TNTDataset(root_dir=root_dir, transform=None)
    
    # Extract labels to use for stratified splitting
    labels = base_dataset.labels

    # Stratified 70/15/15 Split
    # First split: 70% Train, 30% Temporary (Validation + Test)
    train_idx, temp_idx, train_labels, temp_labels = train_test_split(
        np.arange(len(labels)), 
        labels, 
        test_size=0.30, 
        stratify=labels, 
        random_state=seed
    )

    # Second split: Divide the 30% Temporary equally into 15% Validation and 15% Test
    val_idx, test_idx = train_test_split(
        temp_idx, 
        test_size=0.50, 
        stratify=temp_labels, 
        random_state=seed
    )

    # Create subsets using the indices
    train_subset = Subset(base_dataset, train_idx)
    val_subset = Subset(base_dataset, val_idx)
    test_subset = Subset(base_dataset, test_idx)

    # Wrap the subsets to apply the specific transforms
    train_data = TransformSubset(train_subset, transform=train_transform)
    val_data = TransformSubset(val_subset, transform=val_test_transform)
    test_data = TransformSubset(test_subset, transform=val_test_transform)

    # Create DataLoaders with batch size = 32
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    print(f"DataLoaders created for {root_dir}:")
    print(f"  Train: {len(train_data)} samples")
    print(f"  Val:   {len(val_data)} samples")
    print(f"  Test:  {len(test_data)} samples")

    return train_loader, val_loader, test_loader

# --- Execution ---
if __name__ == "__main__":
    # Create DataLoaders for the 512px dataset
    print("Setting up 512x512 DataLoaders...")
    train_loader_512, val_loader_512, test_loader_512 = create_dataloaders('processed_512')

    # Create DataLoaders for the 256px dataset
    print("\nSetting up 256x256 DataLoaders...")
    train_loader_256, val_loader_256, test_loader_256 = create_dataloaders('processed_256')

Setting up 512x512 DataLoaders...
DataLoaders created for processed_512:
  Train: 273 samples
  Val:   59 samples
  Test:  59 samples

Setting up 256x256 DataLoaders...
DataLoaders created for processed_256:
  Train: 1176 samples
  Val:   252 samples
  Test:  252 samples


# Task 3: Load and Modify Pretrained Model

## Step 3.1: Load a Pretrained Model
U-Net model

## Step 3.2: Modify for Binary Classification

## Step 3.3: Freeze Feature Extractor (Phase 1)

# Task 4: Training Infrastructure

## Step 4.1: Loss, Optimizer, and Checkpoint Functions

## Step 4.2: Training and Validation Loops

# Task 5: Three-Phase Training (per patch size)

## Phase 1 - Frozen Features (5 epochs)

## Phase 2 - Resume Training (5 epochs)

## Phase 3 - Fine-Tuning (5 epochs)

# Task 6: Evaluation and Visualization

## Step 6.1: Comprehensive Plots

## Step 6.2: Test Set Evaluation

## Step 6.3: Qualitative Results